In [2]:
!pip3 install librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 4.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 4.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 4.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 4.5 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [librosa]6/17 [librosa]earn]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import librosa
from sklearn.metrics.pairwise import cosine_similarity
import os

# ========== 固定你的文件路径 ==========
csv_path = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/netease_songs_with_id.csv"
audio_folder = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/netease_downloads"

# ========== 13个MFCC 带英文含义表头（作业可直接写报告） ==========
mfcc_cols = [
    "MFCC1_Spectral_Energy_Balance",      # 整体频谱能量平衡
    "MFCC2_Timbre_Brightness",            # 音色明亮度
    "MFCC3_Spectral_Slope_Characteristic",# 频谱斜率特征
    "MFCC4_Harmonic_Structure_1",         # 谐波结构特征1
    "MFCC5_Harmonic_Structure_2",         # 谐波结构特征2
    "MFCC6_Vocal_Resonance_1",           # 人声共振峰特征1
    "MFCC7_Vocal_Resonance_2",           # 人声共振峰特征2
    "MFCC8_Tone_Color_Detail_1",         # 音色细节特征1
    "MFCC9_Tone_Color_Detail_2",         # 音色细节特征2
    "MFCC10_Tone_Color_Detail_3",        # 音色细节特征3
    "MFCC11_Bass_Feature",               # 低频贝斯特征
    "MFCC12_Mid_Frequency_Feature",      # 中频乐器特征
    "MFCC13_High_Frequency_Feature"      # 高频泛音特征
]

# 读取原始干净CSV
df = pd.read_csv(csv_path)

# 存储每首歌13维MFCC向量
mfcc_feature_list = []

print("开始逐首提取13维MFCC音频特征...")
for idx, row in df.iterrows():
    song_name = row["song_name"]
    artist_name = row["artist_name"]
    
    # 拼接音频文件名
    audio_filename = f"{artist_name} - {song_name}.mp3"
    audio_full_path = os.path.join(audio_folder, audio_filename)
    
    try:
        # 加载音频 + 提取13维MFCC
        y, sr = librosa.load(audio_full_path, sr=22050)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean_vec = np.mean(mfcc, axis=1)  # 得到13个数值
        mfcc_feature_list.append(mfcc_mean_vec)
        print(f"成功：{artist_name} - {song_name}")
    except:
        # 音频读取失败填充0
        mfcc_feature_list.append(np.zeros(13))
        print(f"失败：{artist_name} - {song_name}")

# ========== 把13个MFCC特征并入原表格 ==========
mfcc_df = pd.DataFrame(np.array(mfcc_feature_list), columns=mfcc_cols)
df = pd.concat([df, mfcc_df], axis=1)

# ========== 用13维MFCC计算 Novelty ==========
feat_matrix = np.array(mfcc_feature_list)
sim_matrix = cosine_similarity(feat_matrix)
avg_similarity = np.mean(sim_matrix, axis=1)
df["Novelty"] = 1 - avg_similarity

# ========== 直接覆盖原CSV，保存所有内容 ==========
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("\n✅ 全部运行完成！")
print("✅ 已添加：13列带英文释义MFCC + Novelty列")
print("✅ 已直接保存覆盖原CSV文件")
print("\n预览字段：")
print(df[["song_name"] + mfcc_cols[:3] + ["Novelty"]].head())

开始逐首提取13维MFCC音频特征...
成功：郑润泽 - Intro
成功：郑润泽 - Outro
成功：郑润泽 - 任性
成功：郑润泽 - 是你
成功：郑润泽 - 只在今夜
成功：郑润泽 - 像晴天像雨天
成功：郑润泽 - 晚点
成功：郑润泽 - 倔强
成功：郑润泽 - 隐形人
成功：郑润泽 - 一切的一切
失败：郑润泽 - 除了你之外我没喜欢过别人
成功：郑润泽 - 看着我


/var/folders/p6/8w2v2cgx5zjflr0zwpfp9fvh0000gn/T/ipykernel_35527/1767430900.py:45: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_full_path, sr=22050)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


成功：郑润泽 - Crush
成功：郑润泽 - My Dear
成功：郑润泽 - 想悄悄住进你的灵魂
成功：郑润泽 - 雨滴中有你
成功：郑润泽 - 追光自会成为光
成功：郑润泽 - 纯白 (INTRO)
成功：郑润泽 - 纯白
成功：郑润泽 - 万物皆可爱 (INTRO)
成功：郑润泽 - 万物皆可爱
成功：郑润泽 - 结局 (INTRO)
成功：郑润泽 - 结局
成功：郑润泽 - 沙砾 (INTRO)
成功：郑润泽 - 沙砾
成功：郑润泽 - 追光者
成功：郑润泽 - 看见
成功：郑润泽 - 思信
成功：郑润泽 - 看我始终
成功：郑润泽 - 生生
成功：郑润泽 - 凡星
成功：郑润泽 - 小空
成功：郑润泽 - 探心者
成功：郑润泽 - 沉睡花园
成功：郑润泽 - Love is blind (中文版)
成功：郑润泽 - 我们


成功：郑润泽 - Love is blind (英文版)
成功：郑润泽 - 城池
成功：郑润泽 - 生活在别处的你 Another me
成功：郑润泽 - 红莓花儿开
成功：郑润泽 - 别再闹了
成功：郑润泽 - 一江水
成功：郑润泽 - 平凡的一天
成功：郑润泽 - 给你给我
成功：郑润泽 - 哎哟
成功：郑润泽 - 一荤一素
成功：郑润泽 - 南一道街
成功：郑润泽 - 想你想你
成功：郑润泽 - 像我这样的人
成功：郑润泽 - 借
成功：郑润泽 - 盛夏
成功：郑润泽 - 消愁
成功：郑润泽 - 芬芳一生
失败：郑润泽 - 如果有一天我变得很有钱
成功：郑润泽 - Bonus Track：给你给我 (Demo版)
成功：郑润泽 - Bonus Track：一荤一素 (Demo版)
成功：郑润泽 - 意料之中
失败：郑润泽 - 如果有一天我变得很有钱 (必胜客新春版)
成功：陈粒 - 光年之外无名无姓的人
失败：陈粒 - 懒
成功：陈粒 - 绝字句
成功：陈粒 - 广陵赋
成功：陈粒 - 你我
成功：陈粒 - 楔子
成功：陈粒 - 忧伤的大脸猫
成功：陈粒 - 庸人自扰之
成功：陈粒 - 善藏不露
成功：陈粒 - 双目
失败：陈粒 - 浊
成功：陈粒 - 事事
成功：陈粒 - 美梦噩梦
成功：陈粒 - 神的口袋没有后悔药
成功：陈粒 - 从今往后只为你一个人写歌
成功：陈粒 - 悄悄的他
成功：陈粒 - 安眠咒
成功：陈粒 - 多喝热水
成功：陈粒 - 顺平侯
成功：陈粒 - 我说我当不了县长
成功：陈粒 - 舍离书
成功：陈粒 - 盗将行
成功：队长 - 人到中年只剩平凡
成功：队长 - 唯有偏爱才最重要
成功：队长 - 我是双鱼座
成功：队长 - 不如见一面
成功：队长 - Ocean Drop
成功：队长 - 挣脱
失败：队长 - 私人晴天
成功：队长 - 霓虹舞步
失败：队长 - 三百六十五里路
失败：队长 - 之乎者也
成功：队长 - 治愈伤心的旋律
成功：队长 - 独揽狂沙
成功：队长 - 若无你我欲去佗位
成功：队长 - 不爱就不爱
成功：队长 - 不得不爱
成功：队长 - 爱呀恭喜你
失败：队长 - 写一条歌，写你我尔尔
成功：队长 - 容易受伤的女人
成功：队长 - 苏慕遮
成功：队长 - 候鸟
失败：队长 - 计算浪漫
成功：队长 - 日落


In [6]:
import pandas as pd

# 读取你的CSV文件
csv_path = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/netease_songs_with_id.csv"
df = pd.read_csv(csv_path)

# 找到13个MFCC列（假设列名是之前的MFCC1~MFCC13，根据实际列名调整）
mfcc_cols = [col for col in df.columns if col.startswith("MFCC")]

# 筛选出“所有MFCC都是0”的行（即音频加载失败的歌曲）
failed_songs = df[df[mfcc_cols].sum(axis=1) == 0]

print(f"共有 {len(failed_songs)} 首歌音频加载失败，具体如下：")
print(failed_songs[["song_name", "artist_name"]].to_string(index=False))

共有 380 首歌音频加载失败，具体如下：
                                                                    song_name artist_name
                                                                 除了你之外我没喜欢过别人         郑润泽
                                                                  如果有一天我变得很有钱         郑润泽
                                                         如果有一天我变得很有钱 (必胜客新春版)         郑润泽
                                                                            懒          陈粒
                                                                            浊          陈粒
                                                                         私人晴天          队长
                                                                      三百六十五里路          队长
                                                                         之乎者也          队长
                                                                   写一条歌，写你我尔尔          队长
                                                                         计算浪漫 

In [7]:
import pandas as pd
import numpy as np

# 你的文件路径
csv_path = "/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/netease_songs_with_id.csv"
df = pd.read_csv(csv_path)

# 1. 找到所有MFCC列，筛选出有效数据（MFCC不全为0）
mfcc_cols = [col for col in df.columns if col.startswith("MFCC")]
df_valid = df[df[mfcc_cols].sum(axis=1) != 0]  # 有效数据（非0）

# 2. 整体统计
total_valid = len(df_valid)
print(f"📊 整体有效数据量：{total_valid} 首歌（{'✅ 满足≥500' if total_valid>=500 else '❌ 不足500'}）")

# 3. 按音乐风格分组统计
style_stats = []
for style, style_df in df_valid.groupby("music_genre"):
    # 该风格下的音乐人数量
    artist_count = style_df["artist_name"].nunique()
    # 该风格下的总歌曲数
    style_song_count = len(style_df)
    
    # 每个音乐人在该风格下的歌曲数
    artist_song_count = style_df.groupby("artist_name").size().reset_index(name="歌曲数")
    artist_song_str = "\n    - ".join([f"{row['artist_name']}: {row['歌曲数']}首" for _, row in artist_song_count.iterrows()])
    
    style_stats.append({
        "音乐风格": style,
        "音乐人数量": artist_count,
        "有效歌曲总数": style_song_count,
        "各音乐人歌曲数": artist_song_str
    })
    
    # 打印该风格的详细统计
    print(f"\n🎵 音乐风格：{style}")
    print(f"   音乐人数量：{artist_count} 个")
    print(f"   有效歌曲数：{style_song_count} 首")
    print(f"   各音乐人歌曲分布：\n    - {artist_song_str}")

# 4. 保存统计结果到CSV（方便写报告）
stats_df = pd.DataFrame(style_stats)
stats_df.to_csv("/Users/xiahanfei/Desktop/MGS3001-Python-for-Business/AI_music/数据统计结果.csv", 
                index=False, encoding="utf-8-sig")
print("\n✅ 统计结果已保存到：数据统计结果.csv（可直接粘贴到报告）")

📊 整体有效数据量：575 首歌（✅ 满足≥500）

🎵 音乐风格：POP流行
   音乐人数量：7 个
   有效歌曲数：165 首
   各音乐人歌曲分布：
    - h3r3: 24首
    - 万妮达: 16首
    - 薛之谦: 1首
    - 邓紫棋: 1首
    - 郑润泽: 55首
    - 队长: 48首
    - 陈粒: 20首

🎵 音乐风格：R&B
   音乐人数量：5 个
   有效歌曲数：224 首
   各音乐人歌曲分布：
    - 余佳运: 62首
    - 方大同: 7首
    - 王嘉尔: 36首
    - 袁娅维TIA: 32首
    - 裘德: 87首

🎵 音乐风格：嘻哈
   音乐人数量：3 个
   有效歌曲数：186 首
   各音乐人歌曲分布：
    - 刘聪KEY.L: 79首
    - 姜云升: 11首
    - 杨和苏KeyNG: 96首

✅ 统计结果已保存到：数据统计结果.csv（可直接粘贴到报告）
